# FERQUEUE PCV — Eksperimen Training SSD MobileNet V2 FPNLite (`v4 + LAB-CLAHE Halus`) di Google Colab

Notebook ini menjalankan training **SSD MobileNet V2 FPNLite 320×320** dari TFRecord yang sudah diproses **LAB-CLAHE halus** (`clahe_clip_limit = 1.5`, `clahe_grid_size = 8`, kanal luminansi `L` pada ruang warna `LAB`) di laptop lokal, dengan **split dataset dan hyperparameter 100% identik** terhadap baseline `v4` (`train_colab_v4.ipynb`).

### Persiapan di laptop sebelum membuka Colab
1. Bangun TFRecord LAB-CLAHE di laptop lokal (~15 detik, otomatis memakai manifest `v4` 1.083 train / 464 test):
   ```bash
   .venv-pcv/bin/python pcv.py build-records --config apps/computer-vision/configs/coco-v4-clahe.json
   ```
2. Upload isi folder `models/mobilenet/runs/pcv-coco-colab-v4-clahe/` ke Google Drive:
   ```text
   MyDrive/ferqueue/pcv-coco-colab-v4-clahe/
   ├── label_map.pbtxt
   ├── inputs.json
   ├── training-plan.json
   └── records/
       ├── train.tfrecord
       └── test.tfrecord
   ```
3. Aktifkan GPU Colab (**Runtime → Change runtime type → T4 GPU**), lalu jalankan seluruh cell berurutan dari atas ke bawah.

## Cell 1 — Cek GPU, Mount Google Drive, & Salin TFRecord LAB-CLAHE ke SSD Lokal Colab

In [ ]:
import json
import shutil
import subprocess
from pathlib import Path
from google.colab import drive

gpu_check = subprocess.run(["nvidia-smi"], text=True, capture_output=True)
if gpu_check.returncode != 0:
    raise RuntimeError(
        "GPU tidak terdeteksi! Buka menu Runtime -> Change runtime type -> pilih T4 GPU, lalu ulangi Cell 1."
    )
print(gpu_check.stdout)

drive.mount("/content/drive")

DRIVE_RUN_DIR = Path("/content/drive/MyDrive/ferqueue/pcv-coco-colab-v4-clahe")
DRIVE_BASELINE_REPORTS_DIR = Path("/content/drive/MyDrive/ferqueue/pcv-coco-colab-v4/reports")
WORK_DIR = Path("/content/ferqueue_colab_clahe")
VENV_DIR = Path("/content/.venv-pcv")
MODEL_DIR = DRIVE_RUN_DIR / "training"
REPORTS_DIR = DRIVE_RUN_DIR / "reports"

WORK_DIR.mkdir(parents=True, exist_ok=True)
(WORK_DIR / "records").mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)


def find_required(candidates, label):
    for p in candidates:
        if p.is_file():
            return p
    tried = "\n  - ".join(str(p) for p in candidates)
    raise FileNotFoundError(f"File {label} tidak ditemukan. Lokasi yang dicari:\n  - {tried}")


drive_train_tfrec = find_required(
    [DRIVE_RUN_DIR / "records/train.tfrecord", DRIVE_RUN_DIR / "train.tfrecord"],
    "train.tfrecord",
)
drive_test_tfrec = find_required(
    [DRIVE_RUN_DIR / "records/test.tfrecord", DRIVE_RUN_DIR / "test.tfrecord"],
    "test.tfrecord",
)
drive_label_map = find_required([DRIVE_RUN_DIR / "label_map.pbtxt"], "label_map.pbtxt")

LOCAL_TRAIN_TFREC = WORK_DIR / "records/train.tfrecord"
LOCAL_TEST_TFREC = WORK_DIR / "records/test.tfrecord"
LOCAL_LABEL_MAP = WORK_DIR / "label_map.pbtxt"

for src, dst in [
    (drive_train_tfrec, LOCAL_TRAIN_TFREC),
    (drive_test_tfrec, LOCAL_TEST_TFREC),
    (drive_label_map, LOCAL_LABEL_MAP),
]:
    if not dst.is_file() or dst.stat().st_size != src.stat().st_size:
        print(f"Menyalin {src.name} ({src.stat().st_size / (1024**2):.1f} MB) ke SSD lokal Colab...")
        shutil.copy2(src, dst)
    else:
        print(f"Sudah ada di lokal: {dst} ({dst.stat().st_size / (1024**2):.1f} MB)")

# Hyperparameter 100% identik dengan baseline v4 (train_colab_v4.ipynb)
CLASSES = ["Mobil", "Truk", "Bus"]
MODEL_NAME = "ssd_mobilenet_v2_fpnlite_320x320_coco17_tpu-8"
PREPROCESSING = {
    "denoise_kernel": 0,
    "clahe_clip_limit": 1.5,
    "clahe_grid_size": 8,
    "gamma": 1.0,
}

BATCH_SIZE = 8
NUM_STEPS = 4080
LEARNING_RATE = 0.004
FREEZE_BATCHNORM = False
CHECKPOINT_EVERY_N = 100
HORIZONTAL_FLIP = True
BRIGHTNESS_DELTA = 0.1

inputs_file = DRIVE_RUN_DIR / "inputs.json"
if inputs_file.is_file():
    frozen = json.loads(inputs_file.read_text(encoding="utf-8"))
    cfg = frozen.get("config", {})
    if cfg.get("preprocessing"):
        PREPROCESSING = cfg["preprocessing"]
    if float(PREPROCESSING.get("clahe_clip_limit", 0)) <= 0:
        raise ValueError(
            "inputs.json menunjukkan clahe_clip_limit <= 0! Pastikan TFRecord dibuat dengan coco-v4-clahe.json."
        )
    CLASSES = frozen.get("manifest", {}).get("classes", CLASSES)
    BATCH_SIZE = int(cfg.get("batch_size", BATCH_SIZE))
    LEARNING_RATE = float(cfg.get("learning_rate", LEARNING_RATE))
    FREEZE_BATCHNORM = bool(cfg.get("freeze_batchnorm", FREEZE_BATCHNORM))
    CHECKPOINT_EVERY_N = int(cfg.get("checkpoint_every_n", CHECKPOINT_EVERY_N))
    aug = cfg.get("augmentation", {})
    HORIZONTAL_FLIP = bool(aug.get("horizontal_flip", HORIZONTAL_FLIP))
    BRIGHTNESS_DELTA = float(aug.get("brightness_delta", BRIGHTNESS_DELTA))

plan_file = DRIVE_RUN_DIR / "training-plan.json"
if plan_file.is_file():
    plan = json.loads(plan_file.read_text(encoding="utf-8"))
    BATCH_SIZE = int(plan.get("batch_size", BATCH_SIZE))
    NUM_STEPS = int(plan.get("num_steps", NUM_STEPS))
    print("Memuat training-plan.json:", plan)

print("\nPreprocessing aktif (akan ikut disimpan di model.json):", json.dumps(PREPROCESSING))
print(f"Siap! CLASSES={CLASSES}, BATCH_SIZE={BATCH_SIZE}, NUM_STEPS={NUM_STEPS}, LR={LEARNING_RATE}")

## Cell 2 — Siapkan Environment Python 3.11 + TensorFlow 2.15.1 (CUDA) + Object Detection API

In [ ]:
import os
import shutil
import subprocess
import urllib.request
import zipfile
from pathlib import Path

VENV_PY = VENV_DIR / "bin/python"

# 1. Buat venv Python 3.11 dan pasang dependensi training CUDA
subprocess.run(["pip", "install", "-q", "uv"], check=True)
if not VENV_PY.is_file():
    print("Mengunduh Python 3.11 dan membuat venv /content/.venv-pcv ...")
    subprocess.run(["uv", "python", "install", "3.11"], check=True)
    subprocess.run(["uv", "venv", "--python", "3.11", "--seed", str(VENV_DIR)], check=True)

req_file = WORK_DIR / "requirements-colab.txt"
req_file.write_text(
    """Pillow==10.4.0
numpy==1.26.4
opencv-python-headless==4.10.0.84
setuptools==80.9.0
tensorboard==2.15.2
tensorflow[and-cuda]==2.15.1
tf-models-official==2.15.0
tensorflow-text==2.15.0
tensorflow-io==0.36.0
tensorflow-datasets==4.9.3
tensorflow-metadata==1.14.0
keras==2.15.0
tf-keras==2.15.1
protobuf==3.20.3
tf-slim==1.1.0
pycocotools==2.0.7
lvis==0.5.3
lxml==5.3.0
contextlib2==21.6.0
pyparsing==2.4.7
gin-config==0.5.0
PyYAML==6.0.2
six==1.16.0
scipy==1.11.4
matplotlib==3.8.4
pandas==2.1.4
""",
    encoding="utf-8",
)
print("Memasang TensorFlow 2.15.1 (CUDA) & Object Detection API...")
subprocess.run(["uv", "pip", "install", "--python", str(VENV_PY), "-r", str(req_file)], check=True)

# 2. Unduh & compile TensorFlow Object Detection API
TF_MODELS_COMMIT = "40695ad8964145a90ae1abbcc740c086e7364eac"
TF_MODELS_URL = f"https://github.com/tensorflow/models/archive/{TF_MODELS_COMMIT}.zip"
TF_MODELS_DIR = WORK_DIR / "tensorflow-models"
PROTOC_URL = "https://github.com/protocolbuffers/protobuf/releases/download/v3.20.3/protoc-3.20.3-linux-x86_64.zip"
PROTOC_DIR = WORK_DIR / "protoc-3.20.3"

if not (TF_MODELS_DIR / "research/object_detection/protos/pipeline.proto").is_file():
    zip_path = WORK_DIR / "tf-models.zip"
    urllib.request.urlretrieve(TF_MODELS_URL, zip_path)
    extract_tmp = WORK_DIR / "tf-models-extract"
    if extract_tmp.exists():
        shutil.rmtree(extract_tmp)
    with zipfile.ZipFile(zip_path) as zf:
        zf.extractall(extract_tmp)
    if TF_MODELS_DIR.exists():
        shutil.rmtree(TF_MODELS_DIR)
    shutil.move(str(extract_tmp / f"models-{TF_MODELS_COMMIT}"), str(TF_MODELS_DIR))
    shutil.rmtree(extract_tmp, ignore_errors=True)
    zip_path.unlink(missing_ok=True)

protoc_bin = PROTOC_DIR / "bin/protoc"
if not protoc_bin.is_file():
    pzip = WORK_DIR / "protoc.zip"
    urllib.request.urlretrieve(PROTOC_URL, pzip)
    PROTOC_DIR.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(pzip) as zf:
        zf.extractall(PROTOC_DIR)
    protoc_bin.chmod(0o755)
    pzip.unlink(missing_ok=True)

research_dir = TF_MODELS_DIR / "research"
if not (research_dir / "object_detection/protos/pipeline_pb2.py").is_file():
    protos = sorted(str(p) for p in (research_dir / "object_detection/protos").glob("*.proto"))
    subprocess.run(
        [str(protoc_bin), f"--proto_path={research_dir}", f"--python_out={research_dir}", *protos],
        check=True,
    )


def build_venv_env():
    env = os.environ.copy()
    site_packages = list((VENV_DIR / "lib").glob("python3.*/site-packages"))
    nvidia_libs = [str(p) for sp in site_packages for p in sorted(sp.glob("nvidia/*/lib"))]
    nvidia_bins = [str(p) for sp in site_packages for p in sorted(sp.glob("nvidia/*/bin"))]

    python_paths = [str(research_dir), str(research_dir / "slim")]
    if env.get("PYTHONPATH"):
        python_paths.append(env["PYTHONPATH"])
    env["PYTHONPATH"] = ":".join(python_paths)

    ld_paths = nvidia_libs + ([env["LD_LIBRARY_PATH"]] if env.get("LD_LIBRARY_PATH") else [])
    env["LD_LIBRARY_PATH"] = ":".join(ld_paths)

    bin_paths = [str(VENV_DIR / "bin")] + nvidia_bins + ([env["PATH"]] if env.get("PATH") else [])
    env["PATH"] = ":".join(bin_paths)
    env["TF_FORCE_GPU_ALLOW_GROWTH"] = "true"
    env["TF_CPP_MIN_LOG_LEVEL"] = "1"
    return env


def run_venv(args, stream=True):
    env = build_venv_env()
    cmd = [str(VENV_PY), *args]
    if not stream:
        return subprocess.run(cmd, env=env, check=True, text=True, capture_output=True)
    proc = subprocess.Popen(
        cmd,
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    for line in proc.stdout:
        print(line, end="")
    ret = proc.wait()
    if ret != 0:
        raise subprocess.CalledProcessError(ret, cmd)


# 3. Patch tf-slim decoder + verifikasi GPU & jumlah record TFRecord
verify_script = WORK_DIR / "verify_env.py"
verify_script.write_text(
    f"""import importlib.util
from pathlib import Path

spec = importlib.util.find_spec("tf_slim")
decoder_path = Path(spec.origin).parent / "data/tfexample_decoder.py"
src = decoder_path.read_text(encoding="utf-8")
old_imp = "from tensorflow.python.ops import control_flow_ops"
new_imp = "from tensorflow.compat import v1 as control_flow_ops"
if new_imp not in src and old_imp in src:
    decoder_path.write_text(src.replace(old_imp, new_imp), encoding="utf-8")

import tensorflow as tf
gpus = tf.config.list_physical_devices("GPU")
print("TensorFlow:", tf.__version__, "| GPU:", gpus)
if not gpus:
    raise RuntimeError("TensorFlow tidak mendeteksi GPU!")

for split in ("train", "test"):
    p = Path({str(WORK_DIR / "records")!r}) / f"{{split}}.tfrecord"
    n = sum(1 for _ in tf.data.TFRecordDataset(str(p)))
    print(f"TFRecord {{split}}: {{n}} records")
""",
    encoding="utf-8",
)
run_venv([str(verify_script)])

## Cell 3 — Unduh Pretrained SSD MobileNet V2 FPNLite & Buat `pipeline.config`

In [ ]:
import shutil
import tarfile
import urllib.request
from pathlib import Path

PRETRAINED_DIR = WORK_DIR / "pretrained"
PRETRAINED_MODEL_DIR = PRETRAINED_DIR / MODEL_NAME
PRETRAINED_URL = f"https://storage.googleapis.com/download.tensorflow.org/models/object_detection/tf2/20200711/{MODEL_NAME}.tar.gz"
PIPELINE_CONFIG_PATH = WORK_DIR / "pipeline.config"

if not (PRETRAINED_MODEL_DIR / "checkpoint/ckpt-0.index").is_file():
    PRETRAINED_DIR.mkdir(parents=True, exist_ok=True)
    archive = PRETRAINED_DIR / f"{MODEL_NAME}.tar.gz"
    print(f"Mengunduh pretrained {MODEL_NAME}...")
    urllib.request.urlretrieve(PRETRAINED_URL, archive)
    with tarfile.open(archive, "r:gz") as tf_tar:
        tf_tar.extractall(PRETRAINED_DIR)
    archive.unlink(missing_ok=True)

make_cfg_script = WORK_DIR / "make_pipeline.py"
make_cfg_script.write_text(
    f"""from pathlib import Path
from google.protobuf import text_format
from object_detection.protos import pipeline_pb2, train_pb2

pretrained = Path({str(PRETRAINED_MODEL_DIR)!r})
pipeline = pipeline_pb2.TrainEvalPipelineConfig()
text_format.Parse((pretrained / "pipeline.config").read_text(encoding="utf-8"), pipeline)

pipeline.model.ssd.num_classes = {len(CLASSES)}
pipeline.model.ssd.freeze_batchnorm = {FREEZE_BATCHNORM!r}

train = pipeline.train_config
train.batch_size = {BATCH_SIZE}
train.num_steps = {NUM_STEPS}
train.fine_tune_checkpoint = str(pretrained / "checkpoint/ckpt-0")
train.fine_tune_checkpoint_type = "detection"
train.fine_tune_checkpoint_version = train_pb2.CheckpointVersion.V2
train.sync_replicas = False
train.replicas_to_aggregate = 1

train.optimizer.Clear()
momentum = train.optimizer.momentum_optimizer
momentum.momentum_optimizer_value = 0.9
schedule = momentum.learning_rate.cosine_decay_learning_rate
schedule.learning_rate_base = {LEARNING_RATE}
schedule.total_steps = {NUM_STEPS}
schedule.warmup_steps = {min(100, NUM_STEPS // 10)}
schedule.warmup_learning_rate = {LEARNING_RATE / 10}
train.optimizer.use_moving_average = False

del train.data_augmentation_options[:]
if {HORIZONTAL_FLIP!r}:
    train.data_augmentation_options.add().random_horizontal_flip.SetInParent()
if {BRIGHTNESS_DELTA!r}:
    train.data_augmentation_options.add().random_adjust_brightness.max_delta = {BRIGHTNESS_DELTA}

train.max_number_of_boxes = 100

label_map = {str(LOCAL_LABEL_MAP)!r}
pipeline.train_input_reader.label_map_path = label_map
del pipeline.train_input_reader.tf_record_input_reader.input_path[:]
pipeline.train_input_reader.tf_record_input_reader.input_path.append({str(LOCAL_TRAIN_TFREC)!r})
pipeline.train_input_reader.num_readers = 1

for reader in pipeline.eval_input_reader:
    reader.label_map_path = label_map
    reader.shuffle = False
    reader.num_epochs = 1
    reader.num_readers = 1
    del reader.tf_record_input_reader.input_path[:]
    reader.tf_record_input_reader.input_path.append({str(LOCAL_TEST_TFREC)!r})

pipeline.eval_config.use_moving_averages = False
pipeline.eval_config.batch_size = 1

out = Path({str(PIPELINE_CONFIG_PATH)!r})
out.write_text(text_format.MessageToString(pipeline), encoding="utf-8")
print("pipeline.config dibuat di:", out)
""",
    encoding="utf-8",
)
run_venv([str(make_cfg_script)])
shutil.copy2(PIPELINE_CONFIG_PATH, DRIVE_RUN_DIR / "pipeline.config")

## Cell 4 — Training SSD MobileNet V2 FPNLite (Input LAB-CLAHE) di GPU

In [ ]:
print(f"Memulai training (v4 + LAB-CLAHE) hingga {NUM_STEPS} langkah...")
print(f"Checkpoint disimpan di: {MODEL_DIR}")

run_venv([
    "-m", "object_detection.model_main_tf2",
    f"--pipeline_config_path={PIPELINE_CONFIG_PATH}",
    f"--model_dir={MODEL_DIR}",
    f"--num_train_steps={NUM_STEPS}",
    f"--checkpoint_every_n={CHECKPOINT_EVERY_N}",
    "--alsologtostderr",
])
print("\nTraining selesai!")

## Cell 5 — Export `SavedModel` dari Checkpoint Terakhir

In [ ]:
from pathlib import Path

find_ckpt = run_venv(
    ["-c", f"import tensorflow as tf; print(tf.train.latest_checkpoint({str(MODEL_DIR)!r}) or '')"],
    stream=False,
)
latest_ckpt = find_ckpt.stdout.strip().splitlines()[-1].strip()
if not latest_ckpt:
    raise RuntimeError(f"Tidak ditemukan checkpoint di {MODEL_DIR}. Pastikan Cell 4 sudah selesai.")

CKPT_NAME = Path(latest_ckpt).name
EXPORT_DIR = WORK_DIR / "exports" / CKPT_NAME
SAVED_MODEL_DIR = EXPORT_DIR / "saved_model"

print(f"Mengekspor {CKPT_NAME} ke {EXPORT_DIR} ...")
run_venv([
    "-m", "object_detection.exporter_main_v2",
    "--input_type=image_tensor",
    f"--pipeline_config_path={PIPELINE_CONFIG_PATH}",
    f"--trained_checkpoint_dir={MODEL_DIR}",
    f"--output_directory={EXPORT_DIR}",
])
if not (SAVED_MODEL_DIR / "saved_model.pb").is_file():
    raise RuntimeError("Export gagal: saved_model.pb tidak terbentuk.")
print("Export berhasil:", SAVED_MODEL_DIR)

## Cell 6 — Evaluasi COCO pada `test.tfrecord` & Perbandingan Langsung vs Baseline `v4`

In [ ]:
import json
import re
from pathlib import Path

REPORT_PATH = REPORTS_DIR / f"{CKPT_NAME}-test.json"
eval_params_path = WORK_DIR / "eval_params.json"
eval_params_path.write_text(
    json.dumps({
        "classes": CLASSES,
        "saved_model_dir": str(SAVED_MODEL_DIR),
        "test_tfrecord": str(LOCAL_TEST_TFREC),
        "report_path": str(REPORT_PATH),
        "checkpoint": CKPT_NAME,
    }, indent=2),
    encoding="utf-8",
)

eval_script = WORK_DIR / "evaluate_test.py"
eval_script.write_text(
    r"""import copy
import json
import sys
from pathlib import Path
import numpy as np
import tensorflow as tf
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval

params = json.loads(Path(sys.argv[1]).read_text(encoding="utf-8"))
CLASSES = params["classes"]
SAVED_MODEL_DIR = Path(params["saved_model_dir"])
TEST_TFREC = Path(params["test_tfrecord"])
REPORT_PATH = Path(params["report_path"])
CKPT_NAME = params["checkpoint"]

SUMMARY_NAMES = (
    "Precision/mAP", "Precision/mAP@.50IOU", "Precision/mAP@.75IOU",
    "Precision/mAP (small)", "Precision/mAP (medium)", "Precision/mAP (large)",
    "Recall/AR@1", "Recall/AR@10", "Recall/AR@100",
    "Recall/AR@100 (small)", "Recall/AR@100 (medium)", "Recall/AR@100 (large)",
)

feature_spec = {
    "image/height": tf.io.FixedLenFeature([], tf.int64),
    "image/width": tf.io.FixedLenFeature([], tf.int64),
    "image/filename": tf.io.FixedLenFeature([], tf.string),
    "image/encoded": tf.io.FixedLenFeature([], tf.string),
    "image/object/bbox/xmin": tf.io.VarLenFeature(tf.float32),
    "image/object/bbox/xmax": tf.io.VarLenFeature(tf.float32),
    "image/object/bbox/ymin": tf.io.VarLenFeature(tf.float32),
    "image/object/bbox/ymax": tf.io.VarLenFeature(tf.float32),
    "image/object/class/label": tf.io.VarLenFeature(tf.int64),
}

model = tf.saved_model.load(str(SAVED_MODEL_DIR))
detector = model.signatures["serving_default"]

categories = [{"id": idx, "name": name} for idx, name in enumerate(CLASSES, start=1)]
groundtruth = {"info": {}, "images": [], "annotations": [], "categories": categories}
predictions = []
image_count = 0

for image_id, raw in enumerate(tf.data.TFRecordDataset(str(TEST_TFREC)), start=1):
    ex = tf.io.parse_single_example(raw, feature_spec)
    height = int(ex["image/height"].numpy())
    width = int(ex["image/width"].numpy())
    filename = ex["image/filename"].numpy().decode("utf-8", errors="replace")
    img_tensor = tf.io.decode_image(ex["image/encoded"], channels=3, expand_animations=False)
    img_tensor = tf.cast(img_tensor, tf.uint8)[None, ...]

    out = detector(img_tensor)
    count = int(out["num_detections"][0].numpy())
    det_boxes = out["detection_boxes"][0, :count].numpy()
    det_classes = out["detection_classes"][0, :count].numpy()
    det_scores = out["detection_scores"][0, :count].numpy()

    groundtruth["images"].append({"id": image_id, "file_name": filename, "width": width, "height": height})
    xmins = tf.sparse.to_dense(ex["image/object/bbox/xmin"]).numpy()
    xmaxs = tf.sparse.to_dense(ex["image/object/bbox/xmax"]).numpy()
    ymins = tf.sparse.to_dense(ex["image/object/bbox/ymin"]).numpy()
    ymaxs = tf.sparse.to_dense(ex["image/object/bbox/ymax"]).numpy()
    labels = tf.sparse.to_dense(ex["image/object/class/label"]).numpy()

    for x1n, x2n, y1n, y2n, cid in zip(xmins, xmaxs, ymins, ymaxs, labels):
        x1, y1, x2, y2 = float(x1n * width), float(y1n * height), float(x2n * width), float(y2n * height)
        bw, bh = max(0.0, x2 - x1), max(0.0, y2 - y1)
        groundtruth["annotations"].append({
            "id": len(groundtruth["annotations"]) + 1,
            "image_id": image_id,
            "category_id": int(cid),
            "bbox": [x1, y1, bw, bh],
            "area": bw * bh,
            "iscrowd": 0,
        })

    scale = np.array([height, width, height, width], dtype=np.float32)
    for coords, cid, score in zip(det_boxes * scale, det_classes, det_scores):
        ymin, xmin, ymax, xmax = map(float, coords)
        predictions.append({
            "image_id": image_id,
            "category_id": int(cid),
            "bbox": [xmin, ymin, xmax - xmin, ymax - ymin],
            "score": float(score),
        })
    image_count = image_id
    if image_id % 100 == 0:
        print(f"Dievaluasi {image_id} gambar test...")

reference = COCO()
reference.dataset = copy.deepcopy(groundtruth)
reference.createIndex()
if predictions:
    predicted = reference.loadRes(copy.deepcopy(predictions))
else:
    predicted = COCO()
    predicted.dataset = {
        "images": copy.deepcopy(reference.dataset["images"]),
        "categories": copy.deepcopy(reference.dataset["categories"]),
        "annotations": [],
    }
    predicted.createIndex()

evaluator = COCOeval(reference, predicted, iouType="bbox")
evaluator.evaluate()
evaluator.accumulate()
evaluator.summarize()

metrics = {"DetectionBoxes_" + name: float(val) for name, val in zip(SUMMARY_NAMES, evaluator.stats)}
precision = evaluator.eval["precision"]
area_idx = evaluator.params.areaRngLbl.index("all")
max_idx = list(evaluator.params.maxDets).index(100)
names = {cat["id"]: cat["name"] for cat in reference.dataset["categories"]}
for idx, cid in enumerate(evaluator.params.catIds):
    vals = precision[:, :, idx, area_idx, max_idx]
    valid = vals[vals > -1]
    metrics["DetectionBoxes_PerformanceByCategory/mAP/" + names[cid]] = float(np.mean(valid)) if valid.size else -1.0

report = {
    "split": "test",
    "checkpoint": CKPT_NAME,
    "image_count": image_count,
    "metrics": metrics,
}
REPORT_PATH.parent.mkdir(parents=True, exist_ok=True)
REPORT_PATH.write_text(json.dumps(report, indent=2), encoding="utf-8")

print("\n=== RINGKASAN METRIK TEST (v4 + LAB-CLAHE) ===")
print(f"Checkpoint : {CKPT_NAME} ({image_count} gambar)")
print(f"mAP 50:95  : {metrics['DetectionBoxes_Precision/mAP']:.4f}")
print(f"AP@0.50    : {metrics['DetectionBoxes_Precision/mAP@.50IOU']:.4f}")
print(f"AP@0.75    : {metrics['DetectionBoxes_Precision/mAP@.75IOU']:.4f}")
print(f"AR@100     : {metrics['DetectionBoxes_Recall/AR@100']:.4f}")
for cls_name in CLASSES:
    key = f"DetectionBoxes_PerformanceByCategory/mAP/{cls_name}"
    print(f"mAP {cls_name:<6} : {metrics[key]:.4f}")
""",
    encoding="utf-8",
)
run_venv([str(eval_script), str(eval_params_path)])

# Bandingkan langsung dengan laporan baseline v4 di Google Drive (jika ada)
clahe_report = json.loads(REPORT_PATH.read_text(encoding="utf-8"))
baseline_candidates = (
    sorted(
        DRIVE_BASELINE_REPORTS_DIR.glob("ckpt-*-test.json"),
        key=lambda p: int(re.search(r"ckpt-(\d+)", p.name).group(1)) if re.search(r"ckpt-(\d+)", p.name) else -1,
        reverse=True,
    )
    if DRIVE_BASELINE_REPORTS_DIR.is_dir()
    else []
)

if baseline_candidates:
    base_path = baseline_candidates[0]
    base_report = json.loads(base_path.read_text(encoding="utf-8"))
    keys = [
        ("mAP 50:95", "DetectionBoxes_Precision/mAP"),
        ("AP@0.50", "DetectionBoxes_Precision/mAP@.50IOU"),
        ("AP@0.75", "DetectionBoxes_Precision/mAP@.75IOU"),
        ("AR@100", "DetectionBoxes_Recall/AR@100"),
        *[(f"mAP {c}", f"DetectionBoxes_PerformanceByCategory/mAP/{c}") for c in CLASSES],
    ]
    print(f"\n=== PERBANDINGAN: v4 RGB Polos ({base_report['checkpoint']}) vs v4 + LAB-CLAHE ({CKPT_NAME}) ===")
    print(f"{'Metrik':<14} | {'v4 (RGB Polos)':>15} | {'v4 + LAB-CLAHE':>15} | {'Selisih':>10}")
    print("-" * 64)
    comparison_rows = {}
    for label, key in keys:
        v_base = float(base_report["metrics"].get(key, -1.0))
        v_clahe = float(clahe_report["metrics"].get(key, -1.0))
        delta = v_clahe - v_base
        comparison_rows[label] = {"v4": v_base, "v4_clahe": v_clahe, "delta": delta}
        print(f"{label:<14} | {v_base:>15.4f} | {v_clahe:>15.4f} | {delta:>+10.4f}")
    d_map = comparison_rows["mAP 50:95"]["delta"]
    verdict = "LEBIH BAGUS" if d_map > 0.005 else ("LEBIH JELEK" if d_map < -0.005 else "RELATIF SAMA")
    print(f"\nKesimpulan mAP 50:95: {verdict} ({d_map:+.4f})")
    (REPORTS_DIR / "comparison_v4_vs_v4_clahe.json").write_text(
        json.dumps(
            {
                "baseline_report": str(base_path),
                "baseline_checkpoint": base_report["checkpoint"],
                "clahe_checkpoint": CKPT_NAME,
                "preprocessing": PREPROCESSING,
                "verdict": verdict,
                "metrics": comparison_rows,
            },
            indent=2,
        ),
        encoding="utf-8",
    )
else:
    print(f"\nCatatan: Laporan baseline v4 tidak ditemukan di {DRIVE_BASELINE_REPORTS_DIR}; bandingkan manual dengan laporan v4.")

## Cell 7 — Paketkan `pcv-model-colab-v4-clahe-<ckpt>.zip` Siap Impor ke FERQUEUE

In [ ]:
import json
import re
import shutil
import zipfile
from pathlib import Path

ARCHIVE_NAME = f"pcv-model-colab-v4-clahe-{CKPT_NAME}.zip"
LOCAL_ARCHIVE = WORK_DIR / ARCHIVE_NAME
DRIVE_ARCHIVE = DRIVE_RUN_DIR / ARCHIVE_NAME

public_meta = {
    "classes": CLASSES,
    "model": MODEL_NAME,
    "checkpoint": CKPT_NAME,
    "preprocessing": PREPROCESSING,
}

with zipfile.ZipFile(LOCAL_ARCHIVE, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    zf.writestr("model.json", json.dumps(public_meta, ensure_ascii=False, indent=2))
    for path in sorted(SAVED_MODEL_DIR.rglob("*")):
        if path.is_file():
            zf.write(path, "saved_model/" + path.relative_to(SAVED_MODEL_DIR).as_posix())

# Validasi format arsip agar 100% diterima pcv.py import-model
with zipfile.ZipFile(LOCAL_ARCHIVE) as zf:
    names = zf.namelist()
    assert "model.json" in names
    assert "saved_model/saved_model.pb" in names
    assert any(n.startswith("saved_model/variables/") for n in names)
    assert all(n == "model.json" or n.startswith("saved_model/") for n in names)
    meta = json.loads(zf.read("model.json").decode("utf-8"))
    assert re.fullmatch(r"ckpt-\d+", meta["checkpoint"])
    assert meta["model"] == "ssd_mobilenet_v2_fpnlite_320x320_coco17_tpu-8"
    assert float(meta["preprocessing"].get("clahe_clip_limit", 0)) > 0

shutil.copy2(LOCAL_ARCHIVE, DRIVE_ARCHIVE)
print(f"Paket model tersimpan di Google Drive: {DRIVE_ARCHIVE} ({DRIVE_ARCHIVE.stat().st_size / (1024**2):.1f} MB)")
print("\nLangkah impor di laptop lokal FERQUEUE:")
print(f"1. Unduh {ARCHIVE_NAME} dari Google Drive ke outputs/{ARCHIVE_NAME}")
print("2. Jalankan impor (otomatis membawa metadata preprocessing LAB-CLAHE):")
print(f"   .venv-pcv/bin/python pcv.py import-model --config apps/computer-vision/configs/receiver-v4-clahe.json --archive outputs/{ARCHIVE_NAME}")

## Cell 8 — Visualisasi Ekstraksi Fitur Internal MobileNet V2 & FPNLite (`P3`–`P7`)

Membedah secara langsung proses ekstraksi fitur di dalam bobot model hasil trainingmu (`MODEL_DIR`) pada sampel uji `Mobil`, `Truk`, dan `Bus`:
1. **`01_alur_ekstraksi_fitur_per_kelas.png`**: Alur lengkap 1 baris per kelas — `Input (LAB-CLAHE)` → `Conv1 (Tepi)` → `block_3 (Tekstur)` → `block_6 (Komponen)` → `out_relu (Semantik)` → `FPNLite P3–P5 (Overlay)` → `Deteksi Akhir SSD`.
2. **`02_fpnlite_multiskala_p3_p7.png`**: Heatmap aktivasi 5 skala piramida fitur **FPNLite (`P3: 40×40` s/d `P7: 3×3`)** di-overlay di atas foto kendaraan.
3. **`03_channel_terkuat_mobilenetv2.png`**: 8 channel/filter paling aktif pada tahap awal (`Conv1_relu`), menengah (`block_6_expand_relu`), dan dalam (`out_relu`) di backbone MobileNet V2.

Seluruh gambar beresolusi tinggi (200 DPI) langsung ditampilkan di bawah cell ini dan disimpan ke Google Drive (`reports/feature_maps/`).

In [ ]:
import json
from pathlib import Path
from IPython.display import Image as IPyImage, display

FEATURE_MAP_DIR = REPORTS_DIR / "feature_maps"
FEATURE_MAP_DIR.mkdir(parents=True, exist_ok=True)

viz_params_path = WORK_DIR / "feature_viz_params.json"
viz_params_path.write_text(
    json.dumps({
        "classes": CLASSES,
        "pipeline_config": str(PIPELINE_CONFIG_PATH),
        "model_dir": str(MODEL_DIR),
        "test_tfrecord": str(LOCAL_TEST_TFREC),
        "output_dir": str(FEATURE_MAP_DIR),
        "conf_threshold": 0.50,
    }, indent=2),
    encoding="utf-8",
)

viz_script = WORK_DIR / "visualize_mobilenet_features.py"
viz_script.write_text(
    r"""import json
import sys
from pathlib import Path

import cv2
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf
from object_detection.builders import model_builder
from object_detection.utils import config_util

params = json.loads(Path(sys.argv[1]).read_text(encoding="utf-8"))
CLASSES = params["classes"]
PIPELINE_CFG = Path(params["pipeline_config"])
MODEL_DIR = Path(params["model_dir"])
TEST_TFREC = Path(params["test_tfrecord"])
OUT_DIR = Path(params["output_dir"])
CONF_TH = float(params.get("conf_threshold", 0.50))
OUT_DIR.mkdir(parents=True, exist_ok=True)

# 1. Bangun arsitektur SSD MobileNet V2 FPNLite dan pulihkan bobot checkpoint training asli
configs = config_util.get_configs_from_pipeline_file(str(PIPELINE_CFG))
detector = model_builder.build(model_config=configs["model"], is_training=False)
dummy = tf.zeros([1, 320, 320, 3], dtype=tf.float32)
pre_dummy, shape_dummy = detector.preprocess(dummy)
_ = detector.postprocess(detector.predict(pre_dummy, shape_dummy), shape_dummy)

latest_ckpt = tf.train.latest_checkpoint(str(MODEL_DIR))
if not latest_ckpt:
    raise RuntimeError(f"Checkpoint tidak ditemukan di {MODEL_DIR}")
tf.train.Checkpoint(model=detector).restore(latest_ckpt).expect_partial()
ckpt_name = Path(latest_ckpt).name
print(f"Bobot checkpoint dipulihkan: {ckpt_name}")

# 2. Siapkan probe multi-layer backbone MobileNet V2 (Low -> Mid -> High level)
LAYER_SPECS = [
    ("Conv1_relu", "Conv1 (160x160x32)\nTepi & Kontras"),
    ("block_3_expand_relu", "Block 3 (80x80x144)\nTekstur Lokal"),
    ("block_6_expand_relu", "Block 6 / C3 (40x40x192)\nKomponen Bodi"),
    ("block_13_expand_relu", "Block 13 / C4 (20x20x576)\nStruktur Objek"),
    ("out_relu", "Out / C5 (10x10x1280)\nFitur Semantik"),
]
backbone = detector._feature_extractor.classification_backbone
probe_model = tf.keras.Model(
    inputs=backbone.input,
    outputs=[backbone.get_layer(name).output for name, _ in LAYER_SPECS],
)

# 3. Ambil 1 sampel uji representatif untuk setiap kelas (Mobil, Truk, Bus) dari test.tfrecord
feature_spec = {
    "image/filename": tf.io.FixedLenFeature([], tf.string),
    "image/encoded": tf.io.FixedLenFeature([], tf.string),
    "image/object/class/label": tf.io.VarLenFeature(tf.int64),
}
samples_by_class = {}
for raw in tf.data.TFRecordDataset(str(TEST_TFREC)):
    ex = tf.io.parse_single_example(raw, feature_spec)
    labels = tf.sparse.to_dense(ex["image/object/class/label"]).numpy().tolist()
    if not labels:
        continue
    primary_id = int(labels[0])
    if 1 <= primary_id <= len(CLASSES):
        cls_name = CLASSES[primary_id - 1]
        if cls_name not in samples_by_class:
            rgb = tf.io.decode_image(ex["image/encoded"], channels=3, expand_animations=False).numpy()
            fname = Path(ex["image/filename"].numpy().decode("utf-8", errors="replace").replace("\\", "/")).name
            samples_by_class[cls_name] = {"class_name": cls_name, "file_name": fname, "rgb": rgb}
    if len(samples_by_class) == len(CLASSES):
        break

selected_samples = [samples_by_class[c] for c in CLASSES if c in samples_by_class]
if not selected_samples:
    raise RuntimeError("Tidak ada gambar berlabel yang ditemukan di test.tfrecord.")


def norm01(arr: np.ndarray) -> np.ndarray:
    arr = np.asarray(arr, dtype=np.float32)
    lo, hi = float(arr.min()), float(arr.max())
    if hi - lo < 1e-8:
        return np.zeros_like(arr, dtype=np.float32)
    return (arr - lo) / (hi - lo)


def fmap_energy(tensor: tf.Tensor) -> np.ndarray:
    arr = np.abs(tensor[0].numpy())
    return norm01(np.mean(arr, axis=-1))


def overlay_heatmap(rgb: np.ndarray, heat_01: np.ndarray, alpha: float = 0.48) -> np.ndarray:
    h, w = rgb.shape[:2]
    resized = cv2.resize(heat_01, (w, h), interpolation=cv2.INTER_CUBIC)
    cmap_bgr = cv2.applyColorMap(np.clip(resized * 255.0, 0, 255).astype(np.uint8), cv2.COLORMAP_INFERNO)
    cmap_rgb = cv2.cvtColor(cmap_bgr, cv2.COLOR_BGR2RGB)
    blended = (1.0 - alpha) * rgb.astype(np.float32) + alpha * cmap_rgb.astype(np.float32)
    return np.clip(blended, 0, 255).astype(np.uint8)


def draw_detections(rgb: np.ndarray, boxes_yxyx: np.ndarray, scores: np.ndarray, classes_0based: np.ndarray) -> np.ndarray:
    canvas = rgb.copy()
    h, w = canvas.shape[:2]
    colors = {"Mobil": (34, 197, 94), "Truk": (59, 130, 246), "Bus": (249, 115, 22)}
    thick = max(2, int(round(min(h, w) / 160)))
    font_scale = max(0.55, min(h, w) / 620.0)
    shown = 0
    for box, score, cid0 in zip(boxes_yxyx, scores, classes_0based):
        if float(score) < CONF_TH:
            continue
        idx = int(cid0)  # det.postprocess mengembalikan indeks kelas 0-based
        if not 0 <= idx < len(CLASSES):
            continue
        cls_name = CLASSES[idx]
        ymin, xmin, ymax, xmax = map(float, box)
        x1, y1, x2, y2 = int(xmin * w), int(ymin * h), int(xmax * w), int(ymax * h)
        color = colors.get(cls_name, (255, 255, 255))
        cv2.rectangle(canvas, (x1, y1), (x2, y2), color, thick)
        lbl = f"{cls_name} {float(score)*100:.0f}%"
        (tw, th), _ = cv2.getTextSize(lbl, cv2.FONT_HERSHEY_SIMPLEX, font_scale, max(1, thick - 1))
        ly = max(th + 8, y1)
        cv2.rectangle(canvas, (x1, ly - th - 8), (x1 + tw + 8, ly), color, -1)
        cv2.putText(canvas, lbl, (x1 + 4, ly - 4), cv2.FONT_HERSHEY_SIMPLEX, font_scale, (15, 23, 42), max(1, thick - 1), cv2.LINE_AA)
        shown += 1
        if shown >= 5:
            break
    return canvas


# 4. Jalankan forward pass ekstraksi fitur & deteksi untuk setiap sampel kelas
analyzed = []
for item in selected_samples:
    rgb = item["rgb"]
    img_tensor = tf.convert_to_tensor(rgb[None, ...], dtype=tf.float32)
    pre_img, true_shapes = detector.preprocess(img_tensor)
    bb_outs = probe_model(pre_img, training=False)
    fpn_outs = detector._feature_extractor(pre_img)
    pred_dict = detector.predict(pre_img, true_shapes)
    dets = detector.postprocess(pred_dict, true_shapes)
    analyzed.append({
        **item,
        "bb_outs": bb_outs,
        "fpn_outs": fpn_outs,
        "boxes": dets["detection_boxes"][0].numpy(),
        "scores": dets["detection_scores"][0].numpy(),
        "classes0": dets["detection_classes"][0].numpy(),
    })

# Visualisasi 1: Alur Ekstraksi Fitur per Kelas (Input -> Backbone -> FPN Overlay -> Deteksi SSD)
n_rows = len(analyzed)
fig1, axes1 = plt.subplots(n_rows, 7, figsize=(22.0, 3.4 * n_rows))
if n_rows == 1:
    axes1 = np.expand_dims(axes1, axis=0)

col_titles = [
    "1. Input Citra\n(LAB-CLAHE)",
    "2. MobileNetV2\nConv1 (160x160)\nTepi & Kontras",
    "3. MobileNetV2\nBlock 3 (80x80)\nTekstur Lokal",
    "4. MobileNetV2\nBlock 6 (40x40)\nKomponen Bodi",
    "5. MobileNetV2\nOut (10x10)\nFitur Semantik",
    "6. FPNLite\nGabungan P3-P5\n(Overlay)",
    "7. Output SSD\nDeteksi & Kelas\n(conf >= 0.50)",
]

for r, entry in enumerate(analyzed):
    rgb = entry["rgb"]
    h, w = rgb.shape[:2]
    bb = entry["bb_outs"]
    fpn = entry["fpn_outs"]

    # Gabungan energi multi-skala FPNLite P3, P4, P5
    fpn_fused = np.mean(
        [cv2.resize(fmap_energy(fpn[i]), (w, h), interpolation=cv2.INTER_CUBIC) for i in (0, 1, 2)],
        axis=0,
    )
    fpn_overlay = overlay_heatmap(rgb, norm01(fpn_fused), alpha=0.50)
    det_vis = draw_detections(rgb, entry["boxes"], entry["scores"], entry["classes0"])

    panels = [
        (rgb, None),
        (fmap_energy(bb[0]), "inferno"),
        (fmap_energy(bb[1]), "inferno"),
        (fmap_energy(bb[2]), "inferno"),
        (fmap_energy(bb[4]), "inferno"),
        (fpn_overlay, None),
        (det_vis, None),
    ]
    for c, (img_data, cmap) in enumerate(panels):
        ax = axes1[r, c]
        ax.imshow(img_data, cmap=cmap)
        ax.axis("off")
        if r == 0:
            ax.set_title(col_titles[c], fontsize=9.8, fontweight="bold", pad=8)
    axes1[r, 0].text(
        -0.08, 0.5, f"GT: {entry['class_name']}",
        transform=axes1[r, 0].transAxes, rotation=90,
        va="center", ha="center", fontsize=11, fontweight="bold", color="#0F172A",
    )

fig1.suptitle(
    f"Visualisasi Proses Ekstraksi Fitur Internal SSD MobileNet V2 FPNLite ({ckpt_name})",
    fontsize=14, fontweight="bold", y=0.99,
)
plt.tight_layout(rect=[0.02, 0, 1, 0.96])
path_1 = OUT_DIR / "01_alur_ekstraksi_fitur_per_kelas.png"
fig1.savefig(path_1, dpi=200, bbox_inches="tight")
plt.close(fig1)

# Visualisasi 2: Detail 5 Level FPNLite (P3..P7) pada sampel pertama
primary = analyzed[0]
fig2, axes2 = plt.subplots(1, 6, figsize=(20.0, 3.6))
axes2[0].imshow(primary["rgb"])
axes2[0].set_title(f"Input ({primary['class_name']})\n{primary['rgb'].shape[1]}x{primary['rgb'].shape[0]} px", fontsize=9.8, fontweight="bold")
axes2[0].axis("off")

fpn_desc = [
    "P3 (40x40x128)\nObjek Kecil/Jauh",
    "P4 (20x20x128)\nObjek Sedang",
    "P5 (10x10x128)\nObjek Besar",
    "P6 (5x5x128)\nObjek Sangat Besar",
    "P7 (3x3x128)\nKonteks Global",
]
for i in range(5):
    ov = overlay_heatmap(primary["rgb"], fmap_energy(primary["fpn_outs"][i]), alpha=0.52)
    axes2[i + 1].imshow(ov)
    axes2[i + 1].set_title(fpn_desc[i], fontsize=9.5, fontweight="bold")
    axes2[i + 1].axis("off")

fig2.suptitle(
    f"Piramida Fitur Multi-Skala FPNLite (P3-P7) — Sampel {primary['class_name']} ({ckpt_name})",
    fontsize=13, fontweight="bold", y=1.02,
)
plt.tight_layout()
path_2 = OUT_DIR / "02_fpnlite_multiskala_p3_p7.png"
fig2.savefig(path_2, dpi=200, bbox_inches="tight")
plt.close(fig2)

# Visualisasi 3: Top-8 Channel Individual Terkuat di 3 Tahap MobileNet V2
stage_indices = [
    (0, "Conv1_relu (160x160x32) — Filter Tepi & Gradien"),
    (2, "block_6_expand_relu (40x40x192) — Filter Bagian Kendaraan"),
    (4, "out_relu (10x10x1280) — Filter Semantik Kelas"),
]
fig3, axes3 = plt.subplots(len(stage_indices), 8, figsize=(18.5, 7.2))
for r, (bb_idx, stage_title) in enumerate(stage_indices):
    fmap = primary["bb_outs"][bb_idx][0].numpy()  # (H, W, C)
    channel_means = np.mean(np.abs(fmap), axis=(0, 1))
    top_channels = np.argsort(-channel_means)[:8]
    for c, ch_id in enumerate(top_channels):
        ax = axes3[r, c]
        ax.imshow(norm01(fmap[:, :, ch_id]), cmap="viridis")
        ax.set_title(f"Ch #{int(ch_id)}", fontsize=8.8)
        ax.axis("off")
    axes3[r, 0].text(
        -0.15, 0.5, stage_title.split(" — ")[0],
        transform=axes3[r, 0].transAxes, rotation=90,
        va="center", ha="center", fontsize=9.5, fontweight="bold",
    )

fig3.suptitle(
    f"8 Channel Fitur Paling Aktif per Kedalaman MobileNet V2 — Sampel {primary['class_name']} ({ckpt_name})",
    fontsize=13, fontweight="bold", y=0.99,
)
plt.tight_layout(rect=[0.02, 0, 1, 0.95])
path_3 = OUT_DIR / "03_channel_terkuat_mobilenetv2.png"
fig3.savefig(path_3, dpi=200, bbox_inches="tight")
plt.close(fig3)

print("Visualisasi ekstraksi fitur selesai disimpan di:", OUT_DIR)
for p in (path_1, path_2, path_3):
    print(" -", p.name)
""",
    encoding="utf-8",
)

run_venv([str(viz_script), str(viz_params_path)])

# Tampilkan ketiga gambar langsung di output Colab
for fname in [
    "01_alur_ekstraksi_fitur_per_kelas.png",
    "02_fpnlite_multiskala_p3_p7.png",
    "03_channel_terkuat_mobilenetv2.png",
]:
    img_path = FEATURE_MAP_DIR / fname
    if img_path.is_file():
        print(f"\n=== {fname} ===")
        display(IPyImage(filename=str(img_path)))